<div align="center">

#
# 🎬 ActionMesh
### Animated 3D Mesh Generation with Temporal 3D Diffusion

**[Meta Reality Labs](https://ai.facebook.com/research/)** &nbsp;•&nbsp; **[SpAItial](https://www.spaitial.ai/)** &nbsp;•&nbsp; **[University College London](https://geometry.cs.ucl.ac.uk/)**

[Remy Sabathier](https://remysabathier.github.io/RemySabathier/), [David Novotny](https://d-novotny.github.io/), [Niloy J. Mitra](http://www0.cs.ucl.ac.uk/staff/n.mitra/), [Tom Monnier](https://tmonnier.com/)

[![Paper](https://img.shields.io/badge/Paper-ActionMesh-red)](https://arxiv.org/abs/2601.16148)
[![Project Page](https://img.shields.io/badge/Project_Page-green)](https://remysabathier.github.io/actionmesh/)
[![GitHub](https://img.shields.io/badge/GitHub-black)](https://github.com/facebookresearch/actionmesh)
[![HuggingFace](https://img.shields.io/badge/%F0%9F%A4%97%20HuggingFace-Demo-blue)](https://huggingface.co/spaces/facebook/ActionMesh)

<img src="https://github.com/facebookresearch/actionmesh/raw/main/assets/docs/teaser.jpg" width="700">

</div>

---

**ActionMesh** transforms a video into an animated 3D mesh with consistent topology across all frames.

| GPU | Inference Time |
|-----|----------------|
| Colab T4 | ~15 min |
| A100 | ~90 sec |
| H100 | ~45 sec |

⚡ **Requirements**: GPU runtime required (T4 minimum, A100 recommended for faster inference).

## 📦 Setup

### 1. Install ActionMesh

In [4]:
import sys
import torch
pyt_version_str=torch.__version__.split("+")[0].replace(".", "")
version_str="".join([
    f"py3{sys.version_info.minor}_cu",
    torch.version.cuda.replace(".",""),
    f"_pyt{pyt_version_str}"
])
!pip install iopath
!pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{pyt_version_str}/download.html

Looking in links: https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/2110/download.html


In [1]:
%cd /content
!git clone https://github.com/facebookresearch/actionmesh.git
%cd actionmesh
!git submodule update --init --recursive
!pip install -r requirements.txt
!pip install -e .

/content
fatal: destination path 'actionmesh' already exists and is not an empty directory.
/content/actionmesh
Cloning into '/content/actionmesh/third_party/TripoSG'...
Host key verification failed.
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.
fatal: clone of 'git@github.com:VAST-AI-Research/TripoSG.git' into submodule path '/content/actionmesh/third_party/TripoSG' failed
Failed to clone 'third_party/TripoSG'. Retry scheduled
Cloning into '/content/actionmesh/third_party/TripoSG'...
Host key verification failed.
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.
fatal: clone of 'git@github.com:VAST-AI-Research/TripoSG.git' into submodule path '/content/actionmesh/third_party/TripoSG' failed
Failed to clone 'third_party/TripoSG' a second time, aborting
Obtaining file:///content/actionmesh
  Installing build dependencies ... done
  C

In [2]:
if torch.__version__=='1.6.0+cu101' and sys.platform.startswith('linux'):
    !pip install pytorch3d
else:
    !pip install 'git+https://github.com/facebookresearch/pytorch3d.git@stable'






  Cloning https://github.com/facebookresearch/pytorch3d.git (to revision stable) to /tmp/pip-req-build-klq62j_8
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-klq62j_8
  Running command git checkout -q 75ebeeaea0908c5527e7b1e305fbc7681382db47
  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 75ebeeaea0908c5527e7b1e305fbc7681382db47
  Preparing metadata (setup.py) ... done
  Created wheel for pytorch3d: filename=pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl size=65257490 sha256=80ec4e28bcb5a6b549c1a36bbef75fee20ab079e9b5e1fb4d5c79ceea811609c
  Stored in directory: /tmp/pip-ephem-wheel-cache-qspwvsc5/wheels/e9/ec/88/13a99edfc9de29485b221df3503c3bca62e23abac9f2b3a974
Successfully built pytorch3d


In [3]:
import numpy as np

# Check if numpy needs downgrade (Colab has numpy>=2 by default)
if int(np.__version__.split('.')[0]) >= 2:
    print("⚠️ Downgrading numpy to <2 (required by ActionMesh)...")
    !pip install -q "numpy<2"
    print("🔄 Please restart runtime: Runtime > Restart session, then re-run this cell")
else:
    import os
    if not os.path.exists("/content/actionmesh"):
        !git config --global url."https://github.com/".insteadOf "git@github.com:"
        !git clone https://github.com/facebookresearch/actionmesh.git /content/actionmesh
        %cd /content/actionmesh
        !git submodule update --init --recursive
        !pip install -q -r requirements.txt
        !pip install -q -e .
    else:
        %cd /content/actionmesh
        print("✅ ActionMesh already installed!")
    print("✅ Setup complete!")

⚠️ Downgrading numpy to <2 (required by ActionMesh)...
🔄 Please restart runtime: Runtime > Restart session, then re-run this cell


### 2. Install Blender 3.5.1

> **Note**: Blender is only required to export animated `.glb` files.

In [4]:
# @title Install Blender 3.5.1 { display-mode: "form" }
# @markdown Required for generating animated GLB files.

import os

BLENDER_PATH = "/content/blender-3.5.1-linux-x64/blender"

if not os.path.exists(BLENDER_PATH):
    print("📥 Downloading Blender 3.5.1...")
    !wget -q --show-progress -P /content https://download.blender.org/release/Blender3.5/blender-3.5.1-linux-x64.tar.xz
    print("📦 Extracting...")
    !tar -xf /content/blender-3.5.1-linux-x64.tar.xz -C /content
    !rm /content/blender-3.5.1-linux-x64.tar.xz
    print("✅ Blender installed!")
else:
    print("✅ Blender already installed!")

# Verify installation
!{BLENDER_PATH} --version | head -1

📥 Downloading Blender 3.5.1...
blender-3.5.1-linux 100%[===================>] 249.51M  87.8MB/s    in 2.8s    
📦 Extracting...
✅ Blender installed!
Blender 3.5.1


---

## 🚀 Run ActionMesh

**Supported inputs:**
- `.mp4` video file
- Folder containing PNG images
- Number of frames: **16 to 31** (additional frames are ignored)

> 💡 **Tip for custom videos:** For best results, the subject should be isolated on a flat background. We recommend using the [SAM2 demo](https://sam2.metademolab.com/demo) to segment your subject. See our [SAM2 extraction guide](https://github.com/facebookresearch/actionmesh/blob/main/assets/docs/sam2_extraction_guide.md) for detailed instructions.

In [7]:
!ls

actionbench	     CONTRIBUTING.md  outputs		  test.mp4
actionmesh	     inference	      pretrained_weights  third_party
actionmesh.egg-info  LICENSE.txt      pyproject.toml
assets		     notebooks	      README.md
CODE_OF_CONDUCT.md   output-2.glb     requirements.txt


In [8]:
from actionmesh.pipeline_with_3d import ActionMeshPipelineWithMeshInput

ModuleNotFoundError: No module named 'triposg'

In [12]:
%cd /content

# 1. Check if TripoSG exists; if not, clone it fresh
!if [ ! -d "TripoSG" ]; then git clone https://github.com/VAST-AI-Research/TripoSG.git; fi

# 2. Move into the ActionMesh folder
%cd /content/actionmesh

# 3. Run the inference with the TripoSG path securely glued to the front
!PYTHONPATH="/content/TripoSG:$PYTHONPATH" python inference/video_and_3d_to_animated_mesh.py \
  --input test.mp4 \
  --mesh_input output-2.glb \
  --blender_path /content/blender-3.5.1-linux-x64/blender

/content
/content/actionmesh
2026-05-29 04:09:00.098811: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
The config attributes {'embed_frequency': 8, 'embed_include_pi': False, 'embedding_type': 'frequency', 'in_channels': 3, 'latent_channels': 64, 'num_attention_heads': 8, 'num_layers_decoder': 16, 'num_layers_encoder': 8, 'width_decoder': 1024, 'width_encoder': 512} were passed to TripoSGVAE, but are not expected and will be ignored. Please verify your

---

## 🎬 Visualize Results

### Interactive 3D Viewer

In [6]:
# @title Setup 3D Viewer { display-mode: "form" }
# @markdown Run this cell to enable the interactive 3D model viewer.

!pip -q install ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

import base64
from IPython.display import HTML, display
from pathlib import Path

def show_glb(glb_path, width=800, height=500):
    """Display an animated GLB model in an interactive 3D viewer."""
    if not Path(glb_path).exists():
        print(f"❌ File not found: {glb_path}")
        return

    b64 = base64.b64encode(open(glb_path, "rb").read()).decode("utf-8")

    display(HTML(f"""
    <style>
        .viewer-container {{
            border-radius: 12px;
            overflow: hidden;
            box-shadow: 0 4px 20px rgba(0,0,0,0.3);
        }}
    </style>
    <script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
    <div class="viewer-container">
        <model-viewer
            src="data:model/gltf-binary;base64,{b64}"
            style="width:{width}px; height:{height}px; background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);"
            camera-controls
            shadow-intensity="0"
            exposure="1"
            neutral-lighting
            autoplay
            camera-orbit="140deg 80deg 3.5m"
            field-of-view="30deg"
            touch-action="pan-y"
            ar
            ar-modes="webxr scene-viewer quick-look">
        </model-viewer>
    </div>
    """))

print("✅ 3D Viewer ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 105.9 MB/s eta 0:00:00
✅ 3D Viewer ready!


In [7]:
# @title Display Result { display-mode: "form" }
# @markdown View the generated animated mesh.

from pathlib import Path

# Auto-detect output path from input
input_name = Path(input_path).stem
output_dir = f"/content/actionmesh/outputs/{input_name}"
output_glb = f"{output_dir}/animated_mesh.glb"

show_glb(output_glb)

❌ File not found: /content/actionmesh/outputs/davis_camel/animated_mesh.glb


---

<div align="center">

## 📚 Citation

If you find ActionMesh useful, please cite our paper:

```bibtex
@inproceedings{ActionMesh2026,
  author = {Remy Sabathier, David Novotny, Niloy J. Mitra, Tom Monnier},
  title = {ActionMesh: Animated 3D Mesh Generation with Temporal 3D Diffusion},
  year = {2026},
}
```

---

</div>